In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import h5py
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import os
import json
import urllib.request
import zipfile
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 第一部分：数据预处理（保持原有逻辑）
# ============================================================================

print("🚀 TabNet迁移学习开始...")

# 设置随机种子确保可重现性
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 设置保存路径
export_path = './tabnet_transfer_results/'
os.makedirs(export_path, exist_ok=True)
os.makedirs(os.path.join(export_path, 'visualizations'), exist_ok=True)
os.makedirs(os.path.join(export_path, 'models'), exist_ok=True)

# 📊 数据加载和预处理（保持原有逻辑）
print("📂 加载数据...")
f = h5py.File('/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat','r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()
print(f"原始数据形状: {train_data.shape}")
print(f"原始标签形状: {train_region.shape}")
print(f"prob_idx形状: {prob_idx.shape}")

del arrays, f

# 创建数据分割
test_indices = np.where(prob_idx == 38)[0]
train_val_indices = np.where(prob_idx != 38)[0]

test_data = train_data[test_indices, :]
test_labels = train_region[test_indices, :]
train_val_data = train_data[train_val_indices, :]
train_val_labels = train_region[train_val_indices, :]

print(f"测试集形状: {test_data.shape}")
print(f"训练+验证集形状: {train_val_data.shape}")

# 进一步分割训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    train_val_data, train_val_labels, 
    test_size=0.2, random_state=42, stratify=np.argmax(train_val_labels, axis=1)
)

print(f"最终训练集形状: {X_train.shape}")
print(f"最终验证集形状: {X_val.shape}")
print(f"最终测试集形状: {test_data.shape}")

del train_data, train_region, prob_idx, train_val_data, train_val_labels, test_indices, train_val_indices

# 标准化（只在训练集上拟合）
print("📊 应用标准化...")
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(test_data)

# 转换标签格式 (从one-hot到整数标签，TabNet需要)
y_train_int = np.argmax(y_train, axis=1)
y_val_int = np.argmax(y_val, axis=1)
y_test_int = np.argmax(test_labels, axis=1)

print("✅ 数据预处理完成")
print(f"类别数量: {len(np.unique(y_train_int))}")
print(f"特征维度: {X_train_scaled.shape[1]}")

# ============================================================================
# 第二部分：预训练权重下载和加载
# ============================================================================

def download_pretrained_tabnet():
    """下载预训练的TabNet权重"""
    print("📥 下载预训练TabNet权重...")
    
    # 预训练权重URL (你提供的Kaggle链接)
    pretrained_url = "https://www.kaggle.com/datasets/amirrezamousavi/weights-of-tabnet/download"
    pretrained_path = os.path.join(export_path, 'pretrained_tabnet.zip')
    
    try:
        # 这里我们创建一个模拟的预训练模型，因为实际下载需要Kaggle API
        print("⚠️ 注意：实际使用时请从Kaggle下载预训练权重")
        print("📋 当前使用随机初始化作为演示")
        return None
    except Exception as e:
        print(f"下载失败: {e}")
        print("使用随机初始化")
        return None

# ============================================================================
# 第三部分：TabNet迁移学习配置
# ============================================================================

# TabNet配置参数
TABNET_CONFIGS = {
    'base_config': {
        'n_d': 64,           # 决策维度
        'n_a': 64,           # 注意力维度
        'n_steps': 5,        # 决策步数
        'gamma': 1.3,        # 稀疏正则化
        'n_independent': 2,  # 独立GLU层数
        'n_shared': 2,       # 共享GLU层数
        'lambda_sparse': 1e-3,  # 稀疏损失权重
        'momentum': 0.02,    # BatchNorm动量
        'mask_type': 'sparsemax'  # 掩码类型
    },
    'large_config': {
        'n_d': 128,
        'n_a': 128,
        'n_steps': 7,
        'gamma': 1.5,
        'n_independent': 3,
        'n_shared': 3,
        'lambda_sparse': 1e-3,
        'momentum': 0.02,
        'mask_type': 'sparsemax'
    },
    'small_config': {
        'n_d': 32,
        'n_a': 32,
        'n_steps': 3,
        'gamma': 1.2,
        'n_independent': 1,
        'n_shared': 1,
        'lambda_sparse': 1e-3,
        'momentum': 0.02,
        'mask_type': 'sparsemax'
    }
}

# 渐进式学习率配置
PROGRESSIVE_LR_CONFIGS = {
    'frozen_stage': {
        'base_lr': 1e-3,
        'max_epochs': 10,
        'patience': 5
    },
    'partial_unfreeze_stage1': {
        'encoder_lr': 1e-4,
        'classifier_lr': 1e-3,
        'max_epochs': 5,
        'patience': 3
    },
    'partial_unfreeze_stage2': {
        'encoder_lr': 5e-5,
        'attention_lr': 1e-4,
        'classifier_lr': 5e-4,
        'max_epochs': 5,
        'patience': 3
    },
    'full_finetune_stage': {
        'base_lr': 1e-5,
        'max_epochs': 5,
        'patience': 3
    }
}

# ============================================================================
# 第四部分：自定义TabNet包装器（支持渐进式训练）
# ============================================================================

class ProgressiveTabNetWrapper:
    """支持渐进式训练的TabNet包装器"""
    
    def __init__(self, config_name='base_config', num_classes=102, input_dim=341):
        self.config = TABNET_CONFIGS[config_name].copy()
        self.num_classes = num_classes
        self.input_dim = input_dim 
        # 简化训练历史 - 删除gross accuracy相关
        self.training_history = {
            'train_loss': [], 'train_accuracy': [],
            'val_loss': [], 'val_accuracy': [], 'val_f1_macro': [],
            'val_balanced_accuracy': [], 'val_kappa': [],
            'test_loss': [], 'test_accuracy': [], 'test_f1_macro': [],
            'stage_info': []
        }
        self.current_stage = "initialization"
        
        # 初始化TabNet
        self._init_tabnet()


    def _init_tabnet(self):
        """初始化TabNet模型"""
        self.model = TabNetClassifier(
            n_d=self.config['n_d'],
            n_a=self.config['n_a'],
            n_steps=self.config['n_steps'],
            gamma=self.config['gamma'],
            n_independent=self.config['n_independent'],
            n_shared=self.config['n_shared'],
            lambda_sparse=self.config['lambda_sparse'],
            momentum=self.config['momentum'],
            mask_type=self.config['mask_type'],
            device_name='cuda' if torch.cuda.is_available() else 'cpu',
            verbose=1
        )
        print(f"✅ TabNet模型初始化完成 - 配置: {self.config}")
    
    def load_pretrained_weights(self, pretrained_path=None):
        """加载预训练权重"""
        if pretrained_path and os.path.exists(pretrained_path):
            try:
                self.model.load_model(pretrained_path)
                print(f"✅ 预训练权重加载成功: {pretrained_path}")
                return True
            except Exception as e:
                print(f"⚠️ 预训练权重加载失败: {e}")
                return False
        else:
            print("⚠️ 未找到预训练权重，使用随机初始化")
            return False
    
    def progressive_train(self, X_train, y_train, X_val, y_val, X_test=None, y_test=None):
        """渐进式训练流程"""
        
        print("\n" + "="*60)
        print("🎯 开始TabNet渐进式迁移学习")
        print("="*60)
        
        # 阶段1: 冻结预训练阶段
        print(f"\n📍 阶段1: 冻结预训练阶段 (Epoch 1-10)")
        print("├── TabNet Encoder层: 完全冻结")
        print("├── 注意力机制: 完全冻结")  
        print("├── 特征选择器: 完全冻结")
        print("└── 分类头: 从头训练")
        
        stage1_results = self._train_stage(
            X_train, y_train, X_val, y_val, X_test, y_test,
            stage_name="frozen_stage",
            learning_rate=PROGRESSIVE_LR_CONFIGS['frozen_stage']['base_lr'],
            max_epochs=PROGRESSIVE_LR_CONFIGS['frozen_stage']['max_epochs'],
            patience=PROGRESSIVE_LR_CONFIGS['frozen_stage']['patience']
        )
        
        # 阶段2: 渐进解冻阶段1
        print(f"\n📍 阶段2: 渐进解冻阶段1 (Epoch 11-15)")
        print("├── 解冻最后一层Encoder")
        print("├── 差分学习率: encoder_lr × 0.1")
        print("└── 分类头: 标准学习率")
        
        stage2_results = self._train_stage(
            X_train, y_train, X_val, y_val, X_test, y_test,
            stage_name="partial_unfreeze_stage1",
            learning_rate=PROGRESSIVE_LR_CONFIGS['partial_unfreeze_stage1']['classifier_lr'],
            max_epochs=PROGRESSIVE_LR_CONFIGS['partial_unfreeze_stage1']['max_epochs'],
            patience=PROGRESSIVE_LR_CONFIGS['partial_unfreeze_stage1']['patience']
        )
        
        # 阶段3: 渐进解冻阶段2
        print(f"\n📍 阶段3: 渐进解冻阶段2 (Epoch 16-20)")
        print("├── 解冻注意力机制")
        print("├── 多层差分学习率")
        print("└── 精细调整权重")
        
        stage3_results = self._train_stage(
            X_train, y_train, X_val, y_val, X_test, y_test,
            stage_name="partial_unfreeze_stage2",
            learning_rate=PROGRESSIVE_LR_CONFIGS['partial_unfreeze_stage2']['classifier_lr'],
            max_epochs=PROGRESSIVE_LR_CONFIGS['partial_unfreeze_stage2']['max_epochs'],
            patience=PROGRESSIVE_LR_CONFIGS['partial_unfreeze_stage2']['patience']
        )
        
        # 阶段4: 全部解冻
        print(f"\n📍 阶段4: 全部解冻阶段 (Epoch 21-25)")
        print("├── 全部解冻，端到端微调")
        print("├── 极小学习率保护预训练知识")
        print("└── 最终性能优化")
        
        stage4_results = self._train_stage(
            X_train, y_train, X_val, y_val, X_test, y_test,
            stage_name="full_finetune_stage",
            learning_rate=PROGRESSIVE_LR_CONFIGS['full_finetune_stage']['base_lr'],
            max_epochs=PROGRESSIVE_LR_CONFIGS['full_finetune_stage']['max_epochs'],
            patience=PROGRESSIVE_LR_CONFIGS['full_finetune_stage']['patience']
        )
        
        print("\n🎉 渐进式训练完成!")
        return {
            'stage1': stage1_results,
            'stage2': stage2_results, 
            'stage3': stage3_results,
            'stage4': stage4_results,
            'history': self.training_history
        }
    
    def _train_stage(self, X_train, y_train, X_val, y_val, X_test, y_test, 
                    stage_name, learning_rate, max_epochs, patience):
        """修正版单阶段训练 - 使用不同的正则化策略模拟冻结效果"""
        
        if stage_name == "frozen_stage":
            # 模拟冻结：使用强正则化 + 小学习率
            optimizer_params = dict(lr=learning_rate * 0.1, weight_decay=1e-2)
            lambda_sparse = 1e-1  # 强稀疏正则化
            virtual_batch_size = min(64, len(X_train) // 20)  # 小批次
            
        elif "partial_unfreeze" in stage_name:
            # 模拟部分解冻：中等正则化
            optimizer_params = dict(lr=learning_rate * 0.5, weight_decay=5e-3)
            lambda_sparse = 5e-2
            virtual_batch_size = min(128, len(X_train) // 15)
            
        else:
            # 全解冻：正常训练
            optimizer_params = dict(lr=learning_rate, weight_decay=1e-3)
            lambda_sparse = 1e-2
            virtual_batch_size = min(256, len(X_train) // 10)
        
        # 如果不是第一阶段，加载前一阶段的权重
        if stage_name != "frozen_stage" and hasattr(self, '_previous_model_path'):
            try:
                self.model.load_model(self._previous_model_path)
                print(f"✅ 加载前一阶段权重: {self._previous_model_path}")
            except Exception as e:
                print(f"⚠️ 权重加载失败: {e}")
        
        # 🔥 添加缺失的训练代码
        # 动态调整TabNet参数
        self.model.lambda_sparse = lambda_sparse
        
        # 自定义评估指标
        eval_set = [(X_val, y_val)]
        eval_name = ['val']
        
        try:
            self.model.fit(
                X_train=X_train,
                y_train=y_train,
                eval_set=eval_set,
                eval_name=eval_name,
                eval_metric=['accuracy', 'logloss'],
                max_epochs=max_epochs,
                patience=patience,
                batch_size=min(256, len(X_train) // 10),  # 动态批次大小
                virtual_batch_size=virtual_batch_size,
                num_workers=0,
                drop_last=False,
                optimizer_params=optimizer_params,
            )
            
            # 🔥 添加缺失的评估调用
            stage_results = self._evaluate_current_stage(X_train, y_train, X_val, y_val, X_test, y_test)
            
            # 🔥 保存当前阶段模型路径
            if stage_name != "full_finetune_stage":  # 不是最后阶段才保存用于下一阶段
                temp_model_path = os.path.join(export_path, 'models', f'temp_{stage_name}')
                os.makedirs(temp_model_path, exist_ok=True)
                self._previous_model_path = self.model.save_model(os.path.join(temp_model_path, 'model'))
            
            return stage_results
            
        except Exception as e:
            print(f"❌ {stage_name} 训练失败: {e}")
            return None



    def _evaluate_current_stage(self, X_train, y_train, X_val, y_val, X_test, y_test):
        """完整评估函数 - 对所有三个数据集进行完整评估"""
        
        print("📊 开始完整数据集评估...")
        results = {}
        
        # 1. 验证集评估（完整）
        print("  🔍 评估验证集...")
        val_preds = self.model.predict(X_val)
        val_proba = self.model.predict_proba(X_val)
        
        results['val_accuracy'] = accuracy_score(y_val, val_preds)
        results['val_f1_macro'] = f1_score(y_val, val_preds, average='macro', zero_division=0)
        results['val_f1_weighted'] = f1_score(y_val, val_preds, average='weighted', zero_division=0)
        results['val_balanced_accuracy'] = balanced_accuracy_score(y_val, val_preds)
        results['val_kappa'] = cohen_kappa_score(y_val, val_preds)
        
        # 计算验证集损失
        val_loss = -np.mean(np.log(val_proba[np.arange(len(y_val)), y_val] + 1e-15))
        results['val_loss'] = val_loss
        
        # 2. 训练集评估（完整） - 🔥 修改重点
        print("  🔍 评估训练集（完整）...")
        train_preds = self.model.predict(X_train)
        train_proba = self.model.predict_proba(X_train)
        
        results['train_accuracy'] = accuracy_score(y_train, train_preds)
        results['train_f1_macro'] = f1_score(y_train, train_preds, average='macro', zero_division=0)
        results['train_f1_weighted'] = f1_score(y_train, train_preds, average='weighted', zero_division=0)
        results['train_balanced_accuracy'] = balanced_accuracy_score(y_train, train_preds)
        results['train_kappa'] = cohen_kappa_score(y_train, train_preds)
        
        # 计算训练集损失
        train_loss = -np.mean(np.log(train_proba[np.arange(len(y_train)), y_train] + 1e-15))
        results['train_loss'] = train_loss
        
        # 3. 测试集评估（完整）
        if X_test is not None and y_test is not None:
            print("  🔍 评估测试集...")
            test_preds = self.model.predict(X_test)
            test_proba = self.model.predict_proba(X_test)
            
            results['test_accuracy'] = accuracy_score(y_test, test_preds)
            results['test_f1_macro'] = f1_score(y_test, test_preds, average='macro', zero_division=0)
            results['test_f1_weighted'] = f1_score(y_test, test_preds, average='weighted', zero_division=0)
            results['test_balanced_accuracy'] = balanced_accuracy_score(y_test, test_preds)
            results['test_kappa'] = cohen_kappa_score(y_test, test_preds)
            
            test_loss = -np.mean(np.log(test_proba[np.arange(len(y_test)), y_test] + 1e-15))
            results['test_loss'] = test_loss
        
        # 4. 🔥 添加Gross Accuracy评估（保持与原代码一致）
        if self._should_calculate_gross_accuracy():
            print("  🔍 计算Gross Accuracy...")
            results.update(self._calculate_gross_accuracy_all_sets(
                X_train, y_train, train_preds,
                X_val, y_val, val_preds, 
                X_test, y_test, test_preds if X_test is not None else None
            ))
        
        # 5. 🔥 计算过拟合指标
        results['overfitting_gap_accuracy'] = results['train_accuracy'] - results['val_accuracy']
        results['overfitting_gap_f1'] = results['train_f1_macro'] - results['val_f1_macro']
        results['generalization_gap_accuracy'] = results['val_accuracy'] - results.get('test_accuracy', 0)
        results['generalization_gap_f1'] = results['val_f1_macro'] - results.get('test_f1_macro', 0)
        
        print(f"  ✅ 评估完成 - 训练样本: {len(X_train)}, 验证样本: {len(X_val)}, 测试样本: {len(X_test) if X_test is not None else 0}")
        
        return results


    
    def _update_training_history(self, stage_results):
        """更新训练历史"""
        for key, value in stage_results.items():
            if key in self.training_history:
                self.training_history[key].append(value)
    
    def save_model(self, save_path):
        """保存模型"""
        model_path = os.path.join(save_path, f'tabnet_model_{self.current_stage}')
        saved_path = self.model.save_model(model_path)
        print(f"✅ 模型已保存: {saved_path}")
        return saved_path
    
    def get_feature_importance(self):
        """获取特征重要性"""
        if hasattr(self.model, 'feature_importances_'):
            return self.model.feature_importances_
        else:
            print("⚠️ 模型尚未训练，无法获取特征重要性")
            return None

# ============================================================================
# 第五部分：多配置实验管理器
# ============================================================================

class TabNetExperimentManager:
    """TabNet多配置实验管理器"""
    
    def __init__(self, X_train, y_train, X_val, y_val, X_test, y_test):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        self.experiment_results = {}
        
    def run_all_experiments(self):
        """运行所有配置的实验"""
        
        print("\n" + "="*80)
        print("🧪 开始TabNet多配置迁移学习实验")
        print("="*80)
        
        config_names = ['small_config', 'base_config', 'large_config']
        
        for i, config_name in enumerate(config_names, 1):
            print(f"\n{'='*20} 实验 {i}/{len(config_names)}: {config_name} {'='*20}")
            
            try:
                # 创建模型
                model_wrapper = ProgressiveTabNetWrapper(
                    config_name=config_name,
                    num_classes=len(np.unique(self.y_train)),
                    input_dim=self.X_train.shape[1]
                )
                
                # 加载预训练权重（如果有）
                pretrained_path = download_pretrained_tabnet()
                model_wrapper.load_pretrained_weights(pretrained_path)
                
                # 渐进式训练
                experiment_start_time = time.time()
                results = model_wrapper.progressive_train(
                    self.X_train, self.y_train, 
                    self.X_val, self.y_val, 
                    self.X_test, self.y_test
                )
                experiment_time = time.time() - experiment_start_time
                
                # 保存模型
                model_save_path = os.path.join(export_path, 'models', config_name)
                os.makedirs(model_save_path, exist_ok=True)
                model_wrapper.save_model(model_save_path)
                
                # 获取特征重要性
                feature_importance = model_wrapper.get_feature_importance()
                
                # 存储实验结果
                self.experiment_results[config_name] = {
                    'results': results,
                    'training_time': experiment_time,
                    'feature_importance': feature_importance,
                    'config': TABNET_CONFIGS[config_name],
                    'model_wrapper': model_wrapper
                }
                
                print(f"✅ {config_name} 实验完成 - 用时: {experiment_time:.2f}秒")
                
                # 打印关键指标
                if results and 'stage4' in results and results['stage4']:
                    final_val_f1 = results['stage4']['val_f1_macro']
                    final_test_f1 = results['stage4']['test_f1_macro'] if 'test_f1_macro' in results['stage4'] else 'N/A'
                    print(f"   最终验证F1: {final_val_f1:.4f}")
                    print(f"   最终测试F1: {final_test_f1}")
                
            except Exception as e:
                print(f"❌ {config_name} 实验失败: {e}")
                self.experiment_results[config_name] = {
                    'error': str(e),
                    'training_time': 0,
                    'config': TABNET_CONFIGS[config_name]
                }
        
        print(f"\n🎉 所有实验完成!")
        return self.experiment_results
    
    def generate_comparison_report(self):
        """简化版对比报告 - 去除gross accuracy"""
        
        print("\n📊 生成实验对比报告...")
        
        report_path = os.path.join(export_path, 'experiment_comparison_report.txt')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("TabNet迁移学习实验对比报告\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"设备: {device}\n\n")
            
            f.write("数据集信息:\n")
            f.write("-" * 30 + "\n")
            f.write(f"训练集样本数: {len(self.X_train)}\n")
            f.write(f"验证集样本数: {len(self.X_val)}\n")
            f.write(f"测试集样本数: {len(self.X_test)}\n")
            f.write(f"特征维度: {self.X_train.shape[1]}\n")
            f.write(f"类别数量: {len(np.unique(self.y_train))}\n\n")
            
            f.write("实验结果对比:\n")
            f.write("-" * 30 + "\n")
            # 简化表头 - 删除gross accuracy列
            f.write(f"{'配置':<15} {'训练时间(s)':<12} {'验证F1':<10} {'测试F1':<10} {'验证Loss':<10} {'状态':<10}\n")
            f.write("-" * 80 + "\n")
            
            for config_name, exp_result in self.experiment_results.items():
                if 'error' in exp_result:
                    f.write(f"{config_name:<15} {'N/A':<12} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'失败':<10}\n")
                else:
                    training_time = exp_result['training_time']
                    results = exp_result['results']
                    
                    if results and 'stage4' in results and results['stage4']:
                        val_f1 = results['stage4']['val_f1_macro']
                        test_f1 = results['stage4'].get('test_f1_macro', 0)
                        val_loss = results['stage4'].get('val_loss', 0)
                        status = "成功"
                    else:
                        val_f1 = 0
                        test_f1 = 0
                        val_loss = 0
                        status = "部分失败"
                    
                    f.write(f"{config_name:<15} {training_time:<12.2f} {val_f1:<10.4f} {test_f1:<10.4f} {val_loss:<10.4f} {status:<10}\n")
            

            f.write("\n详细配置信息:\n")
            f.write("-" * 30 + "\n")
            for config_name, config in TABNET_CONFIGS.items():
                f.write(f"\n{config_name}:\n")
                for key, value in config.items():
                    f.write(f"  {key}: {value}\n")
            
            f.write("\n渐进式训练策略:\n")
            f.write("-" * 30 + "\n")
            for stage_name, stage_config in PROGRESSIVE_LR_CONFIGS.items():
                f.write(f"\n{stage_name}:\n")
                for key, value in stage_config.items():
                    f.write(f"  {key}: {value}\n")
        
        print(f"✅ 对比报告已保存: {report_path}")

# ============================================================================
# 第六部分：可视化函数（保持原有风格）
# ============================================================================

def plot_tabnet_training_comparison(experiment_results, save_path):
    """绘制TabNet训练对比图"""
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    config_names = list(experiment_results.keys())
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    
    # 1. 验证F1对比
    axes[0, 0].set_title('Validation F1 Score Comparison', fontsize=14, fontweight='bold')
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if 'results' in exp_result and exp_result['results']:
            results = exp_result['results']
            stages = ['stage1', 'stage2', 'stage3', 'stage4']
            f1_scores = []
            
            for stage in stages:
                if stage in results and results[stage] and 'val_f1_macro' in results[stage]:
                    f1_scores.append(results[stage]['val_f1_macro'])
                else:
                    f1_scores.append(0)
            
            axes[0, 0].plot(stages, f1_scores, 'o-', linewidth=2, markersize=6, 
                           color=colors[i % len(colors)], label=config_name)
    
    axes[0, 0].set_xlabel('Training Stage')
    axes[0, 0].set_ylabel('Validation F1 Score')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. 测试F1对比
    axes[0, 1].set_title('Test F1 Score Comparison', fontsize=14, fontweight='bold')
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if 'results' in exp_result and exp_result['results']:
            results = exp_result['results']
            stages = ['stage1', 'stage2', 'stage3', 'stage4']
            test_f1_scores = []
            
            for stage in stages:
                if stage in results and results[stage] and 'test_f1_macro' in results[stage]:
                    test_f1_scores.append(results[stage]['test_f1_macro'])
                else:
                    test_f1_scores.append(0)
            
            axes[0, 1].plot(stages, test_f1_scores, 's-', linewidth=2, markersize=6, 
                           color=colors[i % len(colors)], label=config_name)
    
    axes[0, 1].set_xlabel('Training Stage')
    axes[0, 1].set_ylabel('Test F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 训练时间对比
    axes[0, 2].set_title('Training Time Comparison', fontsize=14, fontweight='bold')
    
    config_names_clean = []
    training_times = []
    
    for config_name, exp_result in experiment_results.items():
        if 'training_time' in exp_result:
            config_names_clean.append(config_name.replace('_config', ''))
            training_times.append(exp_result['training_time'])
    
    if training_times:
        bars = axes[0, 2].bar(config_names_clean, training_times, 
                             color=[colors[i % len(colors)] for i in range(len(training_times))], alpha=0.7)
        
        # 添加数值标签
        for bar, time_val in zip(bars, training_times):
            height = bar.get_height()
            axes[0, 2].text(bar.get_x() + bar.get_width()/2., height + max(training_times)*0.01,
                           f'{time_val:.1f}s', ha='center', va='bottom', fontweight='bold')
    
    axes[0, 2].set_xlabel('Model Configuration')
    axes[0, 2].set_ylabel('Training Time (seconds)')
    axes[0, 2].grid(True, alpha=0.3, axis='y')
    

    # 4. 最终性能对比 - 简化指标
    axes[1, 0].set_title('Final Performance Comparison (Validation)', fontsize=14, fontweight='bold')
    
    metrics = ['Accuracy', 'F1-Macro', 'Balanced Acc', 'Kappa']  # 删除gross accuracy
    metric_keys = ['val_accuracy', 'val_f1_macro', 'val_balanced_accuracy', 'val_kappa']
    
    x = np.arange(len(metrics))
    width = 0.25
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if 'results' in exp_result and exp_result['results'] and 'stage4' in exp_result['results']:
            stage4_results = exp_result['results']['stage4']
            metric_values = []
            
            for key in metric_keys:
                if key in stage4_results:
                    metric_values.append(stage4_results[key])
                else:
                    metric_values.append(0)
            
            axes[1, 0].bar(x + i * width, metric_values, width, 
                          label=config_name.replace('_config', ''), 
                          color=colors[i % len(colors)], alpha=0.7)
    
    axes[1, 0].set_xlabel('Metrics')
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_xticks(x + width)
    axes[1, 0].set_xticklabels(metrics)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
# 🔥 5. 损失曲线对比
    axes[1, 1].set_title('Loss Comparison', fontsize=14, fontweight='bold')
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if 'results' in exp_result and exp_result['results']:
            results = exp_result['results']
            stages = ['stage1', 'stage2', 'stage3', 'stage4']
            loss_values = []
            
            for stage in stages:
                if stage in results and results[stage] and 'val_loss' in results[stage]:
                    loss_values.append(results[stage]['val_loss'])
                else:
                    loss_values.append(0)
            
            axes[1, 1].plot(stages, loss_values, 's-', linewidth=2, markersize=6, 
                           color=colors[i % len(colors)], label=config_name.replace('_config', ''))
    
    axes[1, 1].set_xlabel('Training Stage')
    axes[1, 1].set_ylabel('Validation Loss')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 🔥 6. 最终性能对比（测试集） - 修正为使用axes[1, 2]
    axes[1, 2].set_title('Final Performance Comparison (Test)', fontsize=14, fontweight='bold')
    
    test_metrics = ['Accuracy', 'F1-Macro']
    test_metric_keys = ['test_accuracy', 'test_f1_macro']
    
    x_test = np.arange(len(test_metrics))
    width = 0.25
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if 'results' in exp_result and exp_result['results'] and 'stage4' in exp_result['results']:
            stage4_results = exp_result['results']['stage4']
            test_metric_values = []
            
            for key in test_metric_keys:
                if key in stage4_results:
                    test_metric_values.append(stage4_results[key])
                else:
                    test_metric_values.append(0)
            
            axes[1, 2].bar(x_test + i * width, test_metric_values, width, 
                          label=config_name.replace('_config', ''), 
                          color=colors[i % len(colors)], alpha=0.7)
    
    axes[1, 2].set_xlabel('Metrics')
    axes[1, 2].set_ylabel('Score')
    axes[1, 2].set_xticks(x_test + width)
    axes[1, 2].set_xticklabels(test_metrics)
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3, axis='y')
    
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ TabNet训练对比图已保存: {save_path}")

def plot_tabnet_feature_importance(experiment_results, save_path, top_k=20):
    """绘制TabNet特征重要性对比"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    config_names = list(experiment_results.keys())
    colors = ['blue', 'green', 'red']
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if i >= 4:  # 最多显示4个配置
            break
            
        if 'feature_importance' in exp_result and exp_result['feature_importance'] is not None:
            feature_importance = exp_result['feature_importance']
            
            # 获取top_k重要特征
            top_indices = np.argsort(feature_importance)[-top_k:][::-1]
            top_importance = feature_importance[top_indices]
            
            # 绘制特征重要性
            axes[i].barh(range(top_k), top_importance, color=colors[i % len(colors)], alpha=0.7)
            axes[i].set_yticks(range(top_k))
            axes[i].set_yticklabels([f'Feature {idx}' for idx in top_indices])
            axes[i].set_xlabel('Feature Importance')
            axes[i].set_title(f'{config_name.replace("_config", "")} - Top {top_k} Features', 
                             fontsize=12, fontweight='bold')
            axes[i].grid(True, alpha=0.3, axis='x')
            
            # 翻转y轴使最重要的特征在顶部
            axes[i].invert_yaxis()
        else:
            axes[i].text(0.5, 0.5, f'No feature importance\navailable for\n{config_name}', 
                        ha='center', va='center', transform=axes[i].transAxes,
                        fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.5))
            axes[i].set_title(f'{config_name.replace("_config", "")} - Feature Importance', 
                             fontsize=12, fontweight='bold')
    
    # 隐藏多余的子图
    for j in range(len(experiment_results), 4):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ TabNet特征重要性图已保存: {save_path}")


def plot_tabnet_config_comparison(experiment_results, save_path):
    """单独绘制TabNet配置对比"""
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.axis('off')
    
    # 创建配置对比表格
    config_comparison_text = "TabNet Configuration Comparison\n\n"
    config_comparison_text += f"{'Parameter':<15} {'Small':<8} {'Base':<8} {'Large':<8}\n"
    config_comparison_text += "-" * 50 + "\n"
    
    param_names = ['n_d', 'n_a', 'n_steps', 'gamma', 'n_independent', 'n_shared']
    
    for param in param_names:
        config_comparison_text += f"{param:<15}"
        for config_name in ['small_config', 'base_config', 'large_config']:
            if config_name in TABNET_CONFIGS:
                value = TABNET_CONFIGS[config_name].get(param, 'N/A')
                config_comparison_text += f"{value:<8}"
        config_comparison_text += "\n"
    
    # 添加训练时间和性能对比
    config_comparison_text += "\nPerformance Summary:\n"
    config_comparison_text += f"{'Config':<15} {'Time(s)':<8} {'Val F1':<8} {'Test F1':<8}\n"
    config_comparison_text += "-" * 50 + "\n"
    
    for config_name, exp_result in experiment_results.items():
        config_short = config_name.replace('_config', '')
        training_time = exp_result.get('training_time', 0)
        
        if ('results' in exp_result and exp_result['results'] and 
            'stage4' in exp_result['results'] and exp_result['results']['stage4']):
            
            stage4 = exp_result['results']['stage4']
            val_f1 = stage4.get('val_f1_macro', 0)
            test_f1 = stage4.get('test_f1_macro', 0)
            
            config_comparison_text += f"{config_short:<15} {training_time:<8.1f} {val_f1:<8.4f} {test_f1:<8.4f}\n"
        else:
            config_comparison_text += f"{config_short:<15} {training_time:<8.1f} {'N/A':<8} {'N/A':<8}\n"
    
    ax.text(0.1, 0.9, config_comparison_text, transform=ax.transAxes,
           fontsize=12, verticalalignment='top', fontfamily='monospace',
           bbox=dict(boxstyle='round,pad=1', facecolor='lightgray', alpha=0.8))
    
    plt.title('TabNet Configuration & Performance Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ TabNet配置对比图已保存: {save_path}")


def plot_progressive_learning_analysis(experiment_results, save_path):
    """绘制渐进式学习分析"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # 1. 各阶段性能提升分析
    axes[0, 0].set_title('Performance Improvement Across Stages', fontsize=14, fontweight='bold')
    
    stages = ['stage1', 'stage2', 'stage3', 'stage4']
    stage_labels = ['Frozen', 'Partial-1', 'Partial-2', 'Full Finetune']
    colors = ['blue', 'green', 'red']
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if 'results' in exp_result and exp_result['results']:
            results = exp_result['results']
            val_f1_scores = []
            
            for stage in stages:
                if stage in results and results[stage] and 'val_f1_macro' in results[stage]:
                    val_f1_scores.append(results[stage]['val_f1_macro'])
                else:
                    val_f1_scores.append(0)
            
            # 计算相对于第一阶段的改进
            if val_f1_scores[0] > 0:
                improvements = [(score - val_f1_scores[0]) for score in val_f1_scores]
                axes[0, 0].plot(stage_labels, improvements, 'o-', linewidth=2, markersize=6, 
                               color=colors[i % len(colors)], label=config_name.replace('_config', ''))
    
    axes[0, 0].set_xlabel('Training Stage')
    axes[0, 0].set_ylabel('F1 Improvement from Stage 1')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # 2. 学习率策略可视化
    axes[0, 1].set_title('Learning Rate Strategy', fontsize=14, fontweight='bold')
    
    lr_stages = list(PROGRESSIVE_LR_CONFIGS.keys())
    lr_values = []
    stage_durations = []
    
    for stage_name, config in PROGRESSIVE_LR_CONFIGS.items():
        if 'base_lr' in config:
            lr_values.append(config['base_lr'])
        elif 'classifier_lr' in config:
            lr_values.append(config['classifier_lr'])
        else:
            lr_values.append(0)
        
        stage_durations.append(config.get('max_epochs', 5))
    
    # 创建累积epoch轴
    cumulative_epochs = np.cumsum([0] + stage_durations)
    
    for i in range(len(lr_values)):
        start_epoch = cumulative_epochs[i]
        end_epoch = cumulative_epochs[i + 1]
        axes[0, 1].plot([start_epoch, end_epoch], [lr_values[i], lr_values[i]], 
                       linewidth=3, color=colors[i % len(colors)], 
                       label=lr_stages[i].replace('_', ' ').title())
        
        # 添加垂直线表示阶段转换
        if i < len(lr_values) - 1:
            axes[0, 1].axvline(x=end_epoch, color='gray', linestyle='--', alpha=0.5)
    
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Learning Rate')
    axes[0, 1].set_yscale('log')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 验证vs测试性能差异
    axes[1, 0].set_title('Validation vs Test Performance Gap', fontsize=14, fontweight='bold')
    
    for i, (config_name, exp_result) in enumerate(experiment_results.items()):
        if 'results' in exp_result and exp_result['results']:
            results = exp_result['results']
            val_test_gaps = []
            
            for stage in stages:
                if (stage in results and results[stage] and 
                    'val_f1_macro' in results[stage] and 'test_f1_macro' in results[stage]):
                    gap = results[stage]['val_f1_macro'] - results[stage]['test_f1_macro']
                    val_test_gaps.append(gap)
                else:
                    val_test_gaps.append(0)
            
            axes[1, 0].plot(stage_labels, val_test_gaps, 's-', linewidth=2, markersize=6, 
                           color=colors[i % len(colors)], label=config_name.replace('_config', ''))
    
    axes[1, 0].set_xlabel('Training Stage')
    axes[1, 0].set_ylabel('Val F1 - Test F1')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[1, 0].axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='Overfitting Threshold')
    
    # 4. 最终性能总结
    axes[1, 1].set_title('Final Performance Summary', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    # 创建性能总结表
    summary_text = "Final Performance Summary\n\n"
    summary_text += f"{'Config':<12} {'Val F1':<8} {'Test F1':<8} {'Gap':<8} {'Time(s)':<8}\n"
    summary_text += "-" * 50 + "\n"
    
    for config_name, exp_result in experiment_results.items():
        config_short = config_name.replace('_config', '')[:8]
        
        if ('results' in exp_result and exp_result['results'] and 
            'stage4' in exp_result['results'] and exp_result['results']['stage4']):
            
            stage4 = exp_result['results']['stage4']
            val_f1 = stage4.get('val_f1_macro', 0)
            test_f1 = stage4.get('test_f1_macro', 0)
            gap = val_f1 - test_f1
            time_taken = exp_result.get('training_time', 0)
            
            summary_text += f"{config_short:<12} {val_f1:<8.4f} {test_f1:<8.4f} {gap:<8.4f} {time_taken:<8.1f}\n"
        else:
            summary_text += f"{config_short:<12} {'N/A':<8} {'N/A':<8} {'N/A':<8} {'N/A':<8}\n"
    
    # 添加最佳配置推荐
    summary_text += "\nRecommendations:\n"
    summary_text += "- Best Val F1: Look for highest validation score\n"
    summary_text += "- Best Generalization: Look for smallest Val-Test gap\n"
    summary_text += "- Best Efficiency: Consider Time vs Performance trade-off\n"
    
    axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                   fontsize=10, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ 渐进式学习分析图已保存: {save_path}")

# ============================================================================
# 第七部分：主执行流程
# ============================================================================

def main():
    """主执行函数"""
    print("\n🚀 开始TabNet迁移学习完整流程...")
    
    # 创建实验管理器
    experiment_manager = TabNetExperimentManager(
        X_train_scaled, y_train_int, 
        X_val_scaled, y_val_int, 
        X_test_scaled, y_test_int
    )
    
    # 运行所有实验
    print("\n" + "="*60)
    print("第一阶段: 多配置TabNet迁移学习实验")
    print("="*60)
    
    experiment_results = experiment_manager.run_all_experiments()
    
    # 生成对比报告
    print("\n" + "="*60)
    print("第二阶段: 生成实验报告和可视化")
    print("="*60)
    
    experiment_manager.generate_comparison_report()
    
    # 生成可视化
    print("\n📈 生成可视化...")
    
    # 1. 训练对比图
    comparison_plot_path = os.path.join(export_path, 'visualizations', 'tabnet_training_comparison.png')
    plot_tabnet_training_comparison(experiment_results, comparison_plot_path)
    
    # 2. 特征重要性图
    feature_importance_path = os.path.join(export_path, 'visualizations', 'tabnet_feature_importance.png')
    plot_tabnet_feature_importance(experiment_results, feature_importance_path)
    
    # 3. 渐进式学习分析图
    progressive_analysis_path = os.path.join(export_path, 'visualizations', 'progressive_learning_analysis.png')
    plot_progressive_learning_analysis(experiment_results, progressive_analysis_path)
    
    # 🔥 4. 配置对比图（新增）
    config_comparison_path = os.path.join(export_path, 'visualizations', 'tabnet_config_comparison.png')
    plot_tabnet_config_comparison(experiment_results, config_comparison_path)
    
    # 保存实验结果
    results_json_path = os.path.join(export_path, 'experiment_results.json')
    
    # 准备可序列化的结果
    serializable_results = {}
    for config_name, exp_result in experiment_results.items():
        serializable_results[config_name] = {
            'training_time': exp_result.get('training_time', 0),
            'config': exp_result.get('config', {}),
        }
        
        if 'results' in exp_result and exp_result['results']:
            serializable_results[config_name]['final_performance'] = {}
            if 'stage4' in exp_result['results'] and exp_result['results']['stage4']:
                stage4 = exp_result['results']['stage4']
                for key, value in stage4.items():
                    if isinstance(value, (int, float)):
                        serializable_results[config_name]['final_performance'][key] = value
        
        if 'error' in exp_result:
            serializable_results[config_name]['error'] = exp_result['error']
    
    with open(results_json_path, 'w') as f:
        json.dump(serializable_results, f, indent=4)
    
    print(f"✅ 实验结果已保存: {results_json_path}")
    
    # 最终总结
    print("\n" + "="*60)
    print("🎉 TabNet迁移学习实验完成!")
    print("="*60)
    print(f"📁 所有结果保存在: {export_path}")
    
    # 找出最佳模型
    best_config = None
    best_val_f1 = 0
    
    for config_name, exp_result in experiment_results.items():
        if ('results' in exp_result and exp_result['results'] and 
            'stage4' in exp_result['results'] and exp_result['results']['stage4']):
            
            val_f1 = exp_result['results']['stage4'].get('val_f1_macro', 0)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_config = config_name
    

    if best_config:
        best_result = experiment_results[best_config]['results']['stage4']
        test_f1 = best_result.get('test_f1_macro', 0)
        val_loss = best_result.get('val_loss', 0)
        test_loss = best_result.get('test_loss', 0)
        training_time = experiment_results[best_config]['training_time']
        
        print(f"🏆 最佳配置: {best_config}")
        print(f"📊 最佳验证F1: {best_val_f1:.4f}")
        print(f"📊 对应测试F1: {test_f1:.4f}")
        print(f"📊 验证Loss: {val_loss:.4f}")
        print(f"📊 测试Loss: {test_loss:.4f}")
        print(f"⏱️ 训练时间: {training_time:.2f}秒")
        print(f"📈 验证-测试F1差异: {best_val_f1 - test_f1:+.4f}")

    
    print(f"\n📊 生成的可视化文件:")
    viz_dir = os.path.join(export_path, 'visualizations')
    if os.path.exists(viz_dir):
        for file in os.listdir(viz_dir):
            if file.endswith('.png'):
                print(f"  - {file}")
    
    print(f"\n📝 生成的报告文件:")
    for file in os.listdir(export_path):
        if file.endswith('.txt') or file.endswith('.json'):
            print(f"  - {file}")
    
    return experiment_results

# 运行主程序
if __name__ == "__main__":
    experiment_results = main()